# Splane DLPFC(1/2): Identify uniform spatial domain

## Data preprocessing

In [1]:
from SPACEL.setting import set_environ_seed
seed = 2
set_environ_seed(seed)
from SPACEL import Splane
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib
import anndata
import glob
import os
from natsort import natsorted

Setting environment seed: 2


Here, we will incorporate the cell type composition predicted by **Spoint** into the spatial anndata object for subsequent spatial domain identification in **Splane** using the `add_cell_type_composition` function. This function takes a DataFrame containing the cell type composition matrix as input for spot-based spatial transcriptomic data or a series of cell type annotations as input for single-cell resolution spatial transcriptomic data.

In [2]:
# for loading DLPFC12 data
def load_DLPFC(root_dir='./DLPFC12', section_id='151507'):
    # 151507, ..., 151676
    # 12 in total, 7 layers
    # 151507: n_obs × n_vars = 4221 × 33538
    ad = sc.read_visium(path=os.path.join(root_dir, section_id), count_file=section_id+'_filtered_feature_bc_matrix.h5')
    ad.var_names_make_unique()

    gt_dir = os.path.join(root_dir, section_id, 'gt')
    gt_df = pd.read_csv(os.path.join(gt_dir, 'tissue_positions_list_GTs.txt'), sep=',', header=None, index_col=0)
    ad.obs['original_clusters'] = gt_df.loc[:, 6]
    keep_bcs = ad.obs.dropna().index
    ad = ad[keep_bcs].copy()
    ad.obs['original_clusters'] = ad.obs['original_clusters'].astype(int).astype(str)
    print("spot num", ad)
    return ad

In [ ]:
# Path to the directory
data_dir = './DLPFC'
slices = ['151507', '151508', '151509', '151510']
adata_list = []

for i in range(len(slices)):
    ad = load_DLPFC(data_dir, slices[i])
    adata_list.append(ad)

/hpc/home/fl135/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/hpc/home/fl135/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


spot num AnnData object with n_obs × n_vars = 4221 × 33538
    obs: 'in_tissue', 'array_row', 'array_col', 'original_clusters'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'


/hpc/home/fl135/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/hpc/home/fl135/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


spot num AnnData object with n_obs × n_vars = 4381 × 33538
    obs: 'in_tissue', 'array_row', 'array_col', 'original_clusters'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'


/hpc/home/fl135/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/hpc/home/fl135/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


spot num AnnData object with n_obs × n_vars = 4788 × 33538
    obs: 'in_tissue', 'array_row', 'array_col', 'original_clusters'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'


/hpc/home/fl135/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/hpc/home/fl135/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


spot num AnnData object with n_obs × n_vars = 4595 × 33538
    obs: 'in_tissue', 'array_row', 'array_col', 'original_clusters'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'


In [4]:
ad.obs['original_clusters'].unique()

array(['1', '5', '3', '4', '6', '0', '2'], dtype=object)

In [5]:
# Collect all unique cell types across all .h5ad files
all_cell_types = set()
for adata in adata_list:
    all_cell_types.update(adata.obs['original_clusters'].unique().tolist())

all_cell_types = sorted(list(all_cell_types)) 
# all_cell_types

In [6]:
for i, adata in enumerate(adata_list):    
    # Add cell type composition
    Splane.utils.add_cell_type_composition(adata, celltype_anno=adata.obs['original_clusters'])
    # Add missing cell type columns as 0s
    for ct in all_cell_types:
        if ct not in adata.obs.columns:
            adata.obs[ct] = 0.0
            
    # Reorder columns consistently
    ordered_cols = [c for c in adata.obs.columns if c not in all_cell_types] + all_cell_types
    adata.obs = adata.obs[ordered_cols]
    adata.obs['slice_id'] = slices[i]
    adata.uns['celltypes'] = np.array(all_cell_types, dtype=object)

In [7]:
adata_list[1].obs

,in_tissue,array_row,array_col,original_clusters,0,1,2,3,4,5,6,slice_id
AAACAAGTATCTCCCA-1,1,50,102,3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,151508
AAACAATCTACTAGCA-1,1,3,43,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,151508
AAACACCAATAACTGC-1,1,59,19,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,151508
AAACAGAGCGACTCCT-1,1,14,94,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,151508
AAACAGCTTTCAGAAG-1,1,43,9,6,0.0,0.0,0.0,0.0,0.0,0.0,1.0,151508
...,...,...,...,...,...,...,...,...,...,...,...,...
TTGTTTCACATCCAGG-1,1,58,42,6,0.0,0.0,0.0,0.0,0.0,0.0,1.0,151508
TTGTTTCATTAGTCTA-1,1,60,30,6,0.0,0.0,0.0,0.0,0.0,0.0,1.0,151508
TTGTTTCCATACAACT-1,1,45,27,5,0.0,0.0,0.0,0.0,0.0,1.0,0.0,151508
TTGTTTGTATTACACG-1,1,73,41,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,151508


## Training Splane model

In this step, we initialize the Splane model by ``Splane.init_model(...)`` using the anndata object list as input. The ``n_clusters`` parameter determines the number of spatial domains to be identified. The ``k`` parameter controls the degree of neighbors considered in the model, with a larger ``k`` value resulting in more emphasis on global structure rather than local structure. The ``gnn_dropout`` parameter influences the level of smoothness in the model’s predictions, with a higher ``gnn_dropout`` value resulting in a smoother output that accommodates the sparsity of the spatial transcriptomics data. 

We train the model by ``splane.train(...)`` to obtain latent feature of each spots/cells. The parameter ``d_l`` affects the level of batch effect correction between slices. By default, ``d_l`` is ``0.2`` for spatial transcriptomics data with single cell resolution.

Then, we can identify the spatial domain to which each spot/cell belongs by ``splane.identify_spatial_domain(...)``. By default, the results will be saved in ``spatial_domain`` column in ``.obs``. If the key parameter is provided, the results will be saved in ``.obs[key]``.

In [ ]:
splane_model = Splane.init_model(adata_list, n_clusters=7,use_gpu=False,n_neighbors=25, gnn_dropout=0.5, seed=seed)
splane_model.train(d_l=0.2)
splane_model.identify_spatial_domain()
    
sc.concat(adata_list).write(f'./{seed}/DLPFC_151507_151510_Splane.h5ad')    

Setting global seed: 2
Calculating cell type weights...
Generating GNN inputs...
Calculating largest eigenvalue of normalized graph Laplacian...
Calculating Chebyshev polynomials up to order 2...


/hpc/group/hickeylab/fl135/miniconda3/envs/SPACEL/lib/python3.8/site-packages/SPACEL/Splane/pygcn_utils.py:57: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:641.)
  return torch.sparse.FloatTensor(indices, values, shape)


Setting the weight of similarity loss to 1.210


The best epoch 285 total loss=-2.114 g loss=-1.829 d loss=1.426 d acc=0.318 simi loss=-0.982 db loss=0.362: 100%|██████████| 300/300 [10:19<00:00,  2.06s/it]
/hpc/home/fl135/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [9]:
adata_list[0]

AnnData object with n_obs × n_vars = 4221 × 33538
    obs: 'in_tissue', 'array_row', 'array_col', 'original_clusters', '0', '1', '2', '3', '4', '5', '6', 'slice_id', 'spatial_domain'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial', 'celltypes', 'spatial_neighbors', 'spatial_domain_colors'
    obsm: 'spatial'
    obsp: 'spatial_connectivities', 'spatial_distances'